# 00_setup
Creates the Bronze, Silver, quarantine, schema drift and log tables with explicit schemas. Safe to re-run.

In [0]:
CATALOG, SCHEMA = "workspace", "steam"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE {CATALOG}.{SCHEMA}")

SOURCE_COLS = ["appid", "name", "developer", "publisher", "score_rank", "userscore",
               "positive", "negative", "owners", "average_forever", "average_2weeks",
               "median_forever", "median_2weeks", "price", "initialprice", "discount", "ccu"]
raw_cols = ",\n  ".join(f"{c} STRING" for c in SOURCE_COLS)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS bronze_steam (
  {raw_cols},
  source_file STRING,
  batch_id STRING,
  load_type STRING,
  load_date DATE,
  row_hash STRING,
  load_timestamp TIMESTAMP
) USING DELTA
""")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS silver_steam (
  appid INT,
  name STRING,
  developer STRING,
  publisher STRING,
  positive INT,
  negative INT,
  total_reviews INT,
  approval_rate DOUBLE,
  ccu INT,
  owners_min BIGINT,
  owners_max BIGINT,
  price_usd DOUBLE,
  initialprice_usd DOUBLE,
  discount_pct DOUBLE,
  is_free BOOLEAN,
  row_hash STRING,
  first_load_date DATE,
  last_load_date DATE,
  load_timestamp TIMESTAMP
) USING DELTA
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS silver_quarantine (
  {raw_cols},
  reject_reason STRING,
  load_date DATE,
  load_timestamp TIMESTAMP
) USING DELTA
""")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS schema_drift_log (
  column_name STRING,
  drift_type STRING,
  detected_on DATE,
  source_file STRING,
  load_timestamp TIMESTAMP
) USING DELTA
""")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS pipeline_execution_logs (
  run_id STRING,
  layer STRING,
  load_type STRING,
  file_processed STRING,
  batch_id STRING,
  start_time TIMESTAMP,
  end_time TIMESTAMP,
  status STRING,
  rows_inserted BIGINT,
  rows_updated BIGINT,
  rows_skipped_unchanged BIGINT,
  rows_quarantined BIGINT,
  error_message STRING
) USING DELTA
""")

In [0]:
display(spark.sql("SHOW TABLES"))